[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/intisariapps-com/Intisari-AutoCut-Android/blob/main/AutoCut_Video_Engine_Colab.ipynb)

# 🎬 AutoCut Video AI — Server Komputasi Cloud Colab

> 🚀 **Pusat Komputasi Cloud:** Memproses render video vertikal beresolusi tinggi dengan akselerasi GPU & RAM-Disk
> ⚡ **Cloudflare R2 Global CDN:** Distribusi video ultra-cepat langsung ke galeri HP tanpa leher botol tunnel
> 🛡️ **Anti-Timeout & Asynchronous:** Antrean pekerjaan otomatis yang stabil dan tahan gangguan jaringan
> 📱 **Scan-to-Pairing:** Koneksi instan 1-detik dari aplikasi Android via pemindaian QR Code

Notebook ini berfungsi sebagai **Engine Backend Rendering Video Klip** untuk aplikasi **Intisari AutoCut Android**.

### ✨ Fitur Unggulan Sistem:
1. 📦 **Batch Recipe Serverless**: Unduh master video YouTube 1x, proses otomatis puluhan klip resep sekaligus.
2. 🌐 **Cloudflare R2 High-Speed CDN**: Klip video hasil render langsung dialirkan via Anycast CDN edge berkecepatan tinggi.
3. ⚡ **Producer-Consumer Streaming**: Begitu 1 klip selesai dirender di server, aplikasi Android langsung mengunduhnya secara paralel.
4. 🎙️ **Sub-Second Groq Whisper LPU**: Transkripsi audio kilat tingkat kata (<0.8s per klip) tanpa membebani GPU lokal.
5. 🖥️ **MediaPipe Dynamic Face Tracking**: Pelacakan wajah otomatis berbasis AI untuk reframe vertikal 9:16 yang presisi.
6. 🎨 **Preset Visual Hook Card**: Kartu headline visual otomatis (Breaking News, TikTok Card, Neon, dll.) dengan Pillow RGBA.
7. 🧠 **Model AI Lokal GGUF On-GPU (Qwen2.5)**: Kurasi transkrip klip viral 100% bebas API Key langsung di GPU Colab.
8. 📱 **Scan-to-Pairing (QR Code)**: Tautan server otomatis dikonversi ke gambar QR Code untuk pairing kamera instan dari HP.
9. ⏱️ **Live Countdown Watchdog**: Auto-shutdown runtime saat idle untuk menghemat kuota pemakaian GPU.

---
### 🚀 Cara Menjalankan:
1. Klik tombol **Play (▶)** di sel kode di bawah ini.
2. Tunggu hingga proses setup selesai dan **Tautan Server**, **Spesifikasi Hardware**, serta **QR Code** muncul di layar.
3. Buka aplikasi **Intisari AutoCut Android** di HP Anda -> Buka Tab **Pengaturan** -> Pindai QR Code atau masukkan tautan tersebut.


In [ ]:
"""
🎬 AUTOCUT VIDEO ENGINE — SERVER RENDERING & CLOUDFLARE TUNNEL
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
"""

# @title ⚙️ PUSAT KENDALI ENGINE RENDERING COLAB
# @markdown Atur parameter sesi Colab di bawah ini:
AUTO_SHUTDOWN_MINUTES = 10  # @param [0, 5, 10, 15, 30] {type:"raw"}
GOOGLE_DRIVE_SYNC = "oauth_persistent"  # @param ["oauth_persistent", "native_mount", "oauth_api", "disabled"]
GDRIVE_TOKEN_JSON = ""  # @param {type:"string"}
COOKIE_SOURCE = "gdrive"  # @param ["gdrive", "legacy", "disabled"]
GROQ_API_KEY = ""  # @param {type:"string"}
GEMINI_PSID = ""  # @param {type:"string"}
GEMINI_PSIDTS = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

import os
import sys
import time
import re
import json
import shutil
import zipfile
import subprocess
import sysconfig
import threading
import urllib.request

os.environ["GDRIVE_FOLDER_NAME"] = "AutoCut_Studio/Clips"
if "COOKIE_SOURCE" in globals():
    os.environ["COOKIE_SOURCE"] = str(COOKIE_SOURCE).lower().strip()

print("=" * 80)
print("🚀 MEMULAI AUTOCUT VIDEO ENGINE (BYOC SERVER)")
print("=" * 80)


# Pastikan mount Drive Saya tersedia untuk AutoCut_Studio & Cookies
if "COOKIE_SOURCE" in globals() and str(COOKIE_SOURCE).lower() == "gdrive":
    os.environ["COOKIE_SOURCE"] = "gdrive"
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            print("📁 [Google Drive] Menghubungkan Drive Saya untuk folder AutoCut_Studio...", flush=True)
            from google.colab import drive
            drive.mount("/content/drive")
            print("✅ [Google Drive] Drive Saya terhubung.", flush=True)
        except Exception as e_drv_mnt:
            print(f"⚠️ [Google Drive] Native mount: {e_drv_mnt}", flush=True)

# 0. Integrasi Google Drive (Persistent OAuth untuk testing hilir)
if GOOGLE_DRIVE_SYNC == "oauth_persistent":
    print("🔐 [Drive] Memeriksa kredensial Google Drive...")
    token_json = ""
    # Prioritas 1: Input langsung dari Form Notebook (Sangat mudah bagi pengguna)
    if "GDRIVE_TOKEN_JSON" in globals() and GDRIVE_TOKEN_JSON and GDRIVE_TOKEN_JSON.strip():
        token_json = GDRIVE_TOKEN_JSON.strip()
        print("   📄 Menggunakan token Google Drive langsung dari kolom isian Form Notebook.")
    else:
        # Prioritas 2: Baca dari Colab Secrets
        try:
            from google.colab import userdata
            token_json = userdata.get("GDRIVE_TOKEN_JSON")
            if token_json and token_json.strip():
                print("   🔐 Menggunakan token Google Drive dari brankas Colab Secrets.")
        except Exception:
            token_json = ""

    token_valid = False
    if token_json and token_json.strip():
        try:
            token_data = json.loads(token_json)
            required = ("refresh_token", "token_uri", "client_id")
            missing = [k for k in required if not token_data.get(k)]
            if missing:
                print("   ⚠️ Token JSON tidak lengkap. Field wajib hilang: " + ", ".join(missing))
            else:
                os.environ["GDRIVE_TOKEN_JSON"] = json.dumps(token_data)
os.environ["GDRIVE_FOLDER_NAME"] = "AutoCut_Studio/Clips"
                os.environ.pop("GDRIVE_USE_AUTH_USER", None)
                print("✅ Credential Google Drive terverifikasi. Startup berjalan tanpa popup OAuth.")
                print("   Access token akan di-refresh otomatis memakai refresh_token bila kedaluwarsa.\n")
                token_valid = True
        except Exception as e_tok:
            print(f"   ⚠️ Token tidak valid ({e_tok}).")

    if not token_valid:
        print("🔄 [Auto-Fallback] Menghubungkan Google Drive via Pop-up Izin Langsung...")
        try:
            from google.colab import auth
            auth.authenticate_user()
            os.environ["GDRIVE_USE_AUTH_USER"] = "1"
os.environ["GDRIVE_FOLDER_NAME"] = "AutoCut_Studio/Clips"
            print(f"✅ Google Drive Terhubung (Sesi Aktif): Google Drive > {GDRIVE_FOLDER_NAME}\n")
        except Exception as e_auth:
            print(f"⚠️ Pop-up OAuth tidak selesai ({e_auth}). Mencoba Colab Native Drive Mount...")
            try:
                from google.colab import drive
                drive.mount("/content/drive")
                mount_dir = f"/content/drive/MyDrive/{GDRIVE_FOLDER_NAME.strip('/')}"
                os.makedirs(mount_dir, exist_ok=True)
                os.environ["GDRIVE_MOUNT_PATH"] = mount_dir
                print(f"✅ Google Drive Terhubung via Native Mount: Google Drive > {GDRIVE_FOLDER_NAME}\n")
            except Exception as e_mnt:
                print(f"⚠️ Google Drive dilewati: {e_mnt}\n")
elif GOOGLE_DRIVE_SYNC == "native_mount":
    print("📁 [Drive] Legacy Native Mount aktif (dapat meminta izin lagi pada runtime baru)...")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        mount_dir = f"/content/drive/MyDrive/{GDRIVE_FOLDER_NAME.strip('/')}"
        os.makedirs(mount_dir, exist_ok=True)
        os.environ["GDRIVE_MOUNT_PATH"] = mount_dir
        print(f"✅ Google Drive Terhubung: Google Drive > {GDRIVE_FOLDER_NAME}\n")
    except Exception as e_drv:
        print(f"⚠️ Google Drive mount dilewati atau dibatalkan: {e_drv}\n")
elif GOOGLE_DRIVE_SYNC == "oauth_api":
    print("🔑 [Drive] Legacy Colab OAuth aktif (credential hanya untuk sesi runtime ini)...")
    try:
        from google.colab import auth
        auth.authenticate_user()
        os.environ["GDRIVE_USE_AUTH_USER"] = "1"
os.environ["GDRIVE_FOLDER_NAME"] = "AutoCut_Studio/Clips"
        print(f"✅ Google Drive API sesi aktif: Google Drive > {GDRIVE_FOLDER_NAME}\n")
    except Exception as e_auth:
        print(f"⚠️ Google Drive OAuth dilewati atau dibatalkan: {e_auth}\n")
else:
    print("ℹ️ Mode Langsung: Video dialirkan 100% via Cloudflare Quick Tunnel (Zero-Storage Cost).\n")

# 1. Download binary cloudflared jika belum ada
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("⏳ [1/5] Mengunduh Cloudflare Tunnel client...")
    try:
        subprocess.run([
            "wget", "-q", "-O", "/usr/local/bin/cloudflared",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        ], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
        print("✅ Cloudflare Tunnel siap!")
    except Exception as e_cf:
        print(f"⚠️ Gagal memasang cloudflared: {e_cf}")
else:
    print("✅ Cloudflare Tunnel sudah terpasang.")

# 2. Install dependensi sistem dasar, ImageMagick & fast libraries
print("📦 [2/5] Memeriksa & memasang dependensi sistem (FastAPI, Uvicorn, yt-dlp, curl_cffi, MediaPipe, ImageMagick, llama-cpp-python, Gemini-WebAPI, QRCode, Pydantic)...")
print("   ├─ Memasang ImageMagick OS & font emoji...")
subprocess.run(["apt-get", "install", "-y", "-qq", "imagemagick", "fonts-noto-color-emoji"], check=False)
import shutil
im_bin = shutil.which("magick") or shutil.which("convert")
if not im_bin:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "imagemagick", "fonts-noto-color-emoji"], check=False)
    im_bin = shutil.which("magick") or shutil.which("convert")
if im_bin:
    im_test = subprocess.run([im_bin, "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if "ImageMagick" in im_test.stdout:
        print(f"   ├─ ✨ ImageMagick aktif & terverifikasi: {im_bin}", flush=True)
print("   ├─ Memasang dependensi Web & Media...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "fastapi", "uvicorn[standard]", "python-multipart", "mediapipe", "requests", "qrcode", "pydantic", "gemini_webapi"
], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "yt-dlp", "curl_cffi"], check=False)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "llama-cpp-python", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu122"
], check=False)

# 2.5 Pra-unduh model AI Lokal resmi Qwen2.5 GGUF ke disk NVMe Colab (Eager Warm-Up)
model_cache_dir = "/content/models"
model_target_file = os.path.join(model_cache_dir, "qwen2.5-1.5b-instruct-q4_k_m.gguf")
if not os.path.exists(model_target_file) or os.path.getsize(model_target_file) < 900 * 1024 * 1024:
    os.makedirs(model_cache_dir, exist_ok=True)
    print("🧠 [2.5/5] Mengunduh bobot model AI resmi Qwen2.5-1.5B GGUF (~1.1 GB)...", flush=True)
    gguf_dl_url = "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf"
    try:
        urllib.request.urlretrieve(gguf_dl_url, model_target_file + ".part")
        os.replace(model_target_file + ".part", model_target_file)
        print("✅ Model AI Qwen2.5 berhasil diunduh ke disk Colab!", flush=True)
    except Exception as e_gguf_dl:
        print(f"⚠️ Pra-unduh model AI gagal ({e_gguf_dl}), akan diunduh saat server startup.", flush=True)
else:
    print("✅ Model AI Qwen2.5 sudah tersedia di disk NVMe Colab.", flush=True)

# 2.6 Pra-unduh model Face Tracking MediaPipe resmi (Eager Warm-Up)
face_model_file = os.path.join(model_cache_dir, "face_landmarker.task")
if not os.path.exists(face_model_file) or os.path.getsize(face_model_file) < 100000:
    os.makedirs(model_cache_dir, exist_ok=True)
    print("👤 [2.6/5] Mengunduh model AI Face Tracking MediaPipe (~4.9 MB)...", flush=True)
    face_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    try:
        urllib.request.urlretrieve(face_url, face_model_file + ".part")
        os.replace(face_model_file + ".part", face_model_file)
        print("✅ Model AI Face Tracking MediaPipe siap di disk!", flush=True)
    except Exception as e_face_dl:
        print(f"⚠️ Pra-unduh model Face Tracking gagal ({e_face_dl}), akan diunduh saat runtime.", flush=True)
else:
    print("✅ Model AI Face Tracking MediaPipe sudah tersedia di disk.", flush=True)

# 3. Unduh modul engine biner resmi terenkripsi (Multi-Source Fallback Downloader)
print("📦 [3/5] Menyiapkan modul engine: autocut_video_engine.zip...")
pkg_local = "/tmp/autocut_video_engine.zip"
download_success = False

# Prioritas 1: Jika user mengunggah berkas zip langsung ke /content/ (Pengujian Instan Cepat)
if os.path.exists("/content/autocut_video_engine.zip") and os.path.getsize("/content/autocut_video_engine.zip") > 50000:
    shutil.copyfile("/content/autocut_video_engine.zip", pkg_local)
    download_success = True
    pkg_size_kb = round(os.path.getsize(pkg_local) / 1024, 1)
    print(f"⚡ [LOCAL DETECTED] Menggunakan berkas zip lokal langsung dari /content/ ({pkg_size_kb} KB)!")

if not download_success:
    pkg_candidates = [
        f"https://raw.githubusercontent.com/intisariapps-com/Intisari-AutoCut-Android/main/autocut_video_engine.zip?t={int(time.time())}",
        f"https://raw.githubusercontent.com/intisariapps-com/Intisari-AutoCut-Android/dev/autocut_video_engine.zip?t={int(time.time())}",
        "https://cdn.jsdelivr.net/gh/intisariapps-com/Intisari-AutoCut-Android@main/autocut_video_engine.zip"
    ]
    for candidate_url in pkg_candidates:
        try:
            req = urllib.request.Request(candidate_url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)", "Cache-Control": "no-cache"})
            with urllib.request.urlopen(req, timeout=15) as response, open(pkg_local, "wb") as out_file:
                shutil.copyfileobj(response, out_file)
            if os.path.exists(pkg_local) and zipfile.is_zipfile(pkg_local) and os.path.getsize(pkg_local) > 50000:
                download_success = True
                pkg_size_kb = round(os.path.getsize(pkg_local) / 1024, 1)
                print(f"✅ Paket Biner Berhasil Diunduh ({pkg_size_kb} KB) dari sumber terpercaya.")
                break
        except Exception as e_dl:
            continue

    if not download_success:
        print("⚠️ Mencoba pengunduhan via wget fallback...")
        for candidate_url in pkg_candidates:
            try:
                subprocess.run(["wget", "-q", "--no-cache", "--no-cookies", "-O", pkg_local, candidate_url], check=True)
                if os.path.exists(pkg_local) and zipfile.is_zipfile(pkg_local):
                    download_success = True
                    break
            except Exception:
                continue
if not download_success or not zipfile.is_zipfile(pkg_local):
    raise RuntimeError("❌ GAGAL MENGUNDUH BINER ENGINE! Pastikan repositori Intisari-AutoCut-Android dapat diakses.")

site_pkg = sysconfig.get_paths()["purelib"]
old_engine_dir = os.path.join(site_pkg, "autocut_video_engine")
if os.path.exists(old_engine_dir):
    shutil.rmtree(old_engine_dir, ignore_errors=True)
with zipfile.ZipFile(pkg_local, "r") as zf:
    zf.extractall(site_pkg)

# Bersihkan cache modul Python agar selalu memuat biner terbaru
for mod in list(sys.modules.keys()):
    if "autocut_video_engine" in mod or "pyarmor" in mod or "PIL" in mod:
        del sys.modules[mod]

import autocut_video_engine
eng_ver = getattr(autocut_video_engine, "__version__", "1.4.1")
print(f"✅ Modul Engine AutoCut (Versi: v{eng_ver}) berhasil dipasang di memori!")

# 4. Setel environment & Hugging Face token
if GROQ_API_KEY.strip():
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()
if "GEMINI_PSID" in globals() and GEMINI_PSID.strip():
    os.environ["GEMINI_PSID"] = GEMINI_PSID.strip()
if "GEMINI_PSIDTS" in globals() and GEMINI_PSIDTS.strip():
    os.environ["GEMINI_PSIDTS"] = GEMINI_PSIDTS.strip()
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN.strip()
    print("✅ Hugging Face Terautentikasi (Kuota Unduh Tinggi Aktif!)")

# 5. Unduh & Muat Model AI Whisper ke GPU VRAM di awal (Zero-Delay Warm-Up)
print("⚡ [4/5] Menginisialisasi AI Engine (Groq Whisper Cloud LPU & MediaPipe Face Tracking)...")
print("🌐 [5/5] Meluncurkan server backend pada port 8000...")
import uvicorn
from autocut_video_engine.server import app as fastapi_app

def start_uvicorn():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(2.5)

# 5. Jalankan Cloudflare Tunnel dan ambil URL publik
print("🚇 Membuka Cloudflare Quick Tunnel...")
subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False)

cf_log_path = "/tmp/cloudflared.log"
with open(cf_log_path, "w", encoding="utf-8") as f_init:
    pass

cf_log_file = open(cf_log_path, "w+", encoding="utf-8")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.DEVNULL,
    stderr=cf_log_file,
    text=True
)

public_tunnel_url = None
timeout_sec = 30
start_t = time.time()
with open(cf_log_path, "r", encoding="utf-8", errors="ignore") as f:
    while time.time() - start_t < timeout_sec:
        line = f.readline()
        if not line:
            time.sleep(0.2)
            continue
        if "trycloudflare.com" in line:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                public_tunnel_url = match.group(0)
                break

if not public_tunnel_url:
    print("⚠️ Gagal mendapatkan URL Cloudflare otomatis. Periksa koneksi.")
else:
    # Ambil telemetri hardware server dari health endpoint lokal
    telemetry_data = {}
    try:
        import urllib.request

os.environ["GDRIVE_FOLDER_NAME"] = "AutoCut_Studio/Clips"
if "COOKIE_SOURCE" in globals():
    os.environ["COOKIE_SOURCE"] = str(COOKIE_SOURCE).lower().strip()
        with urllib.request.urlopen("http://127.0.0.1:8000/api/v1/health", timeout=3) as h_res:
            h_json = json.loads(h_res.read().decode("utf-8"))
            telemetry_data = h_json.get("telemetry", {})
    except Exception:
        pass

    gpu_name = telemetry_data.get("gpu", {}).get("name", "Tidak Terdeteksi")
    gpu_vram = telemetry_data.get("gpu", {}).get("vram_total_mb", 0)
    cpu_model = telemetry_data.get("cpu", {}).get("model", "Multi-Core")
    cpu_cores = telemetry_data.get("cpu", {}).get("cores", 2)
    ram_total = telemetry_data.get("memory", {}).get("ram_total_mb", 0)
    ram_disk_mb = telemetry_data.get("ram_disk", {}).get("total_mb", 0)
    encoder = telemetry_data.get("capabilities", {}).get("video_encoder", "libx264")
    benchmark = telemetry_data.get("capabilities", {}).get("benchmark_estimate", "45-60s/klip")

    print("\n" + "=" * 80)
    print("🎉 SERVER AUTOCUT VIDEO ENGINE BERHASIL ONLINE!")
    print("=" * 80)
    print(f"👉 URL TUNNEL PUBLIK : {public_tunnel_url}")
    print("-" * 80)
    print("🖥️  PROFIL SPESIFIKASI SERVER COLAB:")
    print(f"   • GPU Hardware       : {gpu_name} ({gpu_vram} MB VRAM)")
    print(f"   • CPU Processor      : {cpu_model} ({cpu_cores} Cores)")
    print(f"   • RAM Sistem         : {ram_total} MB")
    print(f"   • RAM-Disk (/dev/shm): {ram_disk_mb} MB (Buffer Cepat 4.000 MB/s)")
    print(f"   • Encoder Video      : {encoder.upper()} (Akselerasi Aktif)")
    print(f"   • Estimasi Kecepatan : {benchmark}")
    print("=" * 80)

    # Tampilkan QR Code di layar Colab untuk pairing kamera smartphone
    try:
        import qrcode
        from IPython.display import display, Image
        import io
        qr = qrcode.QRCode(box_size=7, border=2)
        qr.add_data(public_tunnel_url)
        qr.make(fit=True)
        img = qr.make_image(fill_color="black", back_color="white")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        print("\n📱 PINDAI QR CODE DI BAWAH INI DARI APLIKASI ANDROID:")
        display(Image(buf.getvalue()))
    except Exception as e_qr:
        print(f"(QR Code viewer fallback: {e_qr})")

    print("\nℹ️ Server siap menerima instruksi render video dari aplikasi Android!")
    print("⏱️ Tekan tombol Stop (⏹) kapan saja untuk mematikan server.")

# 6. Live Interactive Watchdog Timer (ATM dari intiVoice V1.6.3)
def _idle_watchdog_loop():
    try:
        from autocut_video_engine.server import watchdog
    except ImportError:
        try:
            from autocut_video_engine.watchdog import watchdog
        except ImportError:
            from colab.engine.watchdog import watchdog
    timeout_min = float(AUTO_SHUTDOWN_MINUTES)
    if timeout_min <= 0:
        print("⏱️ [AUTO-SHUTDOWN NONAKTIF] Mesin akan standby tanpa batas waktu.")
        while True:
            time.sleep(1)
        return

    idle_limit_sec = timeout_min * 60.0
    display_handle = None
    try:
        from IPython.display import display, HTML
        init_html = (
            "<div style='font-family: monospace; font-size: 13px; color: #00D2B4; background: #071952; "
            "padding: 10px 16px; border-radius: 10px; border: 1px solid #1A73E8; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
            "⏳ <b>COUNTDOWN AUTO-SHUTDOWN:</b> Menyiapkan timer realtime..."
            "</div>"
        )
        display_handle = display(HTML(init_html), display_id=True)
    except Exception:
        display_handle = None

    while True:
        time.sleep(1.0)
        try:
            is_active = getattr(watchdog, "is_busy", getattr(watchdog, "active_jobs", 0) > 0)
            if is_active:
                watchdog.touch()

            elapsed_idle = time.time() - watchdog.last_activity
            remaining_sec = max(0, int(idle_limit_sec - elapsed_idle))
            mins = remaining_sec // 60
            secs = remaining_sec % 60

            if display_handle:
                try:
                    if is_active:
                        widget_html = (
                            "<div style='font-family: monospace; font-size: 13px; color: #FFD700; background: #1a1a00; "
                            "padding: 10px 16px; border-radius: 10px; border: 1px solid #FFA500; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
                            "⚡ <b>ENGINE SEDANG MERENDER:</b> Memproses video klip... <span style='color: #00D2B4;'>(Timer idle dijeda)</span>"
                            "</div>"
                        )
                    else:
                        badge_color = "#00D2B4" if remaining_sec > 60 else ("#FFA500" if remaining_sec > 30 else "#FF4444")
                        bg_color = "#071952" if remaining_sec > 60 else ("#2b1700" if remaining_sec > 30 else "#2b0000")
                        border_color = "#1A73E8" if remaining_sec > 60 else ("#FF8C00" if remaining_sec > 30 else "#FF0000")

                        widget_html = (
                            f"<div style='font-family: monospace; font-size: 13px; color: {badge_color}; background: {bg_color}; "
                            f"padding: 10px 16px; border-radius: 10px; border: 1px solid {border_color}; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.3);'>"
                            f"⏳ <b>COUNTDOWN AUTO-SHUTDOWN:</b> Sisa Waktu Idle: <b style='font-size: 16px; color: #FFFFFF;'>{mins:02d}:{secs:02d}</b> "
                            f"<span style='font-size: 11px; opacity: 0.8;'>| Reset otomatis tiap ada job render baru</span>"
                            f"</div>"
                        )
                    from IPython.display import HTML
                    display_handle.update(HTML(widget_html))
                except Exception:
                    pass

            if elapsed_idle >= idle_limit_sec and not is_active:
                print(f"\n🛑 [AUTO-SHUTDOWN] TIDAK ADA AKTIVITAS RENDER SELAMA {int(timeout_min)} MENIT.")
                print("💡 Memutuskan runtime Google Colab untuk menghemat kuota compute units...")
                try:
                    from google.colab import runtime
                    runtime.unassign()
                except Exception:
                    os._exit(0)
                return
        except Exception:
            pass

try:
    _idle_watchdog_loop()
except KeyboardInterrupt:
    print("\n🛑 Sesi server dihentikan oleh pengguna.")
